In [0]:
from pyspark.sql import functions as F

In [0]:
# Definindo o caminho do volume onde estão os datasets
volume_path = "/Volumes/workspace/default/olist_files"

In [0]:
# teste inicial, lendo o dataset de clientes (customers)
# lendo o dataset
df_customers = spark.read.csv(
    f"{volume_path}/olist_customers_dataset.csv", # caminho completo do arquivo, juntando a pasta do volume com o nome específico do arquivo
    header=True, # Informa ao Spark que a primeira linha do arquivo CSV contém o cabeçalho, evitando que seja tratada como se fosse um dado/registro comum.
    inferSchema=True # Diz para o Spark analisar os dados automaticamente para descobrir o tipo de cada coluna
)

In [0]:
# Mostrando as primeiras linhas e a estrutura do DataFrame
display(df_customers.limit(5)) # mostrando as primeiras 5 linhas
# estrutura do DataFrame
df_customers.printSchema() # imprimindo o DataFrame na tela

In [0]:
# Datasets normais
datasets = [
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_orders_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation"
]

In [0]:

# 1. Criando as tabelas Bronze normais

for dataset in datasets:

    file_path = f"{volume_path}/{dataset}.csv"
    table_name = f"bronze_{dataset}"

    df = spark.read.csv(
        file_path,
        header=True,
        inferSchema=True
    )

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

    print(f"Tabela Bronze criada: {table_name}")


In [0]:
# 2. Criando Reviews separadamente

file_path = f"{volume_path}/olist_order_reviews_dataset.csv"

df_reviews = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(file_path)
)

df_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_olist_order_reviews_dataset")

print("Tabela Bronze criada: bronze_olist_order_reviews_dataset")